In [1]:
import ee 
from GEE_ImageFusion import *
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
ee.Authenticate()
ee.Initialize(project='landsat-fusion')

In [3]:
landsat_stat_paths = ["LT05_Metadata_Export.csv", "LE07_Metadata_Export.csv", "LC08_Metadata_Export.csv"]

dtype_mappping = dict(WRS_PATH='int32', WRS_ROW='int32', CLOUD_COVER='int32')


landsat_df = pd.concat([pd.read_csv(p).assign(source=p.split('_')[0]).astype(dtype_mappping) for p in landsat_stat_paths])

landsat_df['date'] = pd.to_datetime(landsat_df['date'])

landsat_df

,system_index,date,WRS_PATH,WRS_ROW,CLOUD_COVER,IMAGE_QUALITY,source
0,LT05_191024_20000317,2000-03-17,191,24,55,9.0,LT05
1,LT05_191024_20000504,2000-05-04,191,24,33,9.0,LT05
2,LT05_191024_20000605,2000-06-05,191,24,31,9.0,LT05
3,LT05_191024_20000621,2000-06-21,191,24,1,9.0,LT05
4,LT05_191024_20000909,2000-09-09,191,24,55,9.0,LT05
...,...,...,...,...,...,...,...
3249,LC08_198025_20170925,2017-09-25,198,25,18,NaN,LC08
3250,LC08_198025_20171027,2017-10-27,198,25,35,NaN,LC08
3251,LC08_198025_20171112,2017-11-12,198,25,81,NaN,LC08
3252,LC08_198025_20171128,2017-11-28,198,25,36,NaN,LC08


In [4]:
relevant_tiles = gpd.read_file("relevant_tiles.gpkg").astype({k: dtype_mappping[k] for k in dtype_mappping if k != "CLOUD_COVER"})
relevant_tiles

,WRS_PATH,WRS_ROW,remaining_area,geometry
0,195,27,0.244297,MULTIPOLYGON Z (((1058648.357 6114844.043 7000...
1,194,27,0.425767,MULTIPOLYGON Z (((1230636.971 6114844.043 7000...
2,193,27,0.464225,MULTIPOLYGON Z (((1402625.584 6114844.043 7000...
3,192,27,0.263979,MULTIPOLYGON Z (((1441934.958 6143902.698 7000...
4,196,26,0.328061,"MULTIPOLYGON Z (((954787.273 6352934.763 7000,..."
5,195,26,0.827768,MULTIPOLYGON Z (((1126775.886 6352934.763 7000...
6,194,26,1.000000,MULTIPOLYGON Z (((1298764.499 6352934.763 7000...
7,193,26,0.976172,MULTIPOLYGON Z (((1398333.958 6369279.357 7000...
8,192,26,0.512568,MULTIPOLYGON Z (((1386543.758 6410757.198 7000...
9,197,25,0.413723,"MULTIPOLYGON Z (((854377.092 6597448.438 7000,..."


In [5]:
relevant_landsat_df = landsat_df.merge(relevant_tiles, how='inner', on=['WRS_PATH', 'WRS_ROW'])


In [6]:
def get_borders_for_tile_and_year(data, wrs_path, wrs_row, year, cloud_cover_limit, date_format='%Y-%m-%d'):

    df = data[(data["WRS_PATH"] == wrs_path) & (data["WRS_ROW"] == wrs_row) & (data['CLOUD_COVER'] < cloud_cover_limit)]

    sorted_df = df.sort_values(by='date')

    start_date = pd.to_datetime(sorted_df[sorted_df['date'].dt.year < year]['date'].tail(1).item()) - pd.Timedelta(days=1)


    end_date = pd.to_datetime(sorted_df[sorted_df['date'].dt.year > year]['date'].head(1).item()) + pd.Timedelta(days=1)

    return start_date.strftime(date_format), end_date.strftime(date_format)

In [7]:
comp_cfg = LandsatFusionComputationConfig(
    common_bands=["blue", "green", "red"], modis_interp_sample_rate=10
)

wrs_row = 23
wrs_path = 195
year = 2003
cloud_cover_limit = 20

In [8]:
start_day, end_day = get_borders_for_tile_and_year(relevant_landsat_df, wrs_path=wrs_path, wrs_row=wrs_row, year=year, cloud_cover_limit=cloud_cover_limit)

In [9]:
img_coll = compute_year_for_tile(
    wrs_path=wrs_path,
    wrs_row=wrs_row,
    start_date=start_day,
    end_date=end_day,
    year=2003,
    fusion_comp_cfg=comp_cfg
)

In [10]:
img_coll.size().getInfo()

EEException: Collection.toList: Empty date ranges not supported for the current operation.